In [6]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(42)


In [18]:
from SLM import SmallLanguageModel
weight_path = 'shapely_weights.pth'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
state = torch.load(weight_path, map_location=device)

model2 = SmallLanguageModel()
model2.load_state_dict(state)
model2.to(device)
model2.eval()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/shapely/data/shakespeare.txt'

In [ ]:
# Generate
idx = torch.zeros((1,1),dtype=torch.long, device=device)
message = decode(model.generate(idx, max_new_tokens=200)[0].tolist())
print(message)

In [ ]:
path = '/content/drive/MyDrive/shapely/data/shakespeare.txt'
with open(path,'r',encoding='utf-8') as f:
    text = f.read()

print('Character length of dataset:',len(text))

chars = sorted(list(set(text)))
vocab_size = len(chars)

# Create char,i encodings
itoc = {i:c for i,c in enumerate(chars)}
ctoi = {c:i for i,c in enumerate(chars)}
encode = lambda x: [ctoi[c] for c in x]
decode = lambda x: ''.join([itoc[i] for i in x])

# Encode and Split Text
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
test_data = data[n:]

In [ ]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, _ = x.shape
        k = self.key(x)    # B,T,C
        q = self.query(x)  # B,T,C
        v = self.value(x)  # B,T,C

        headsize = k.size(-1)
        wei = q @ k.transpose(-2, -1) * headsize**-.5 # B,T,C @ B,C,T = B,T,T
        
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B,T,T)
        wei = torch.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ v # B,T,T @ B,T,C = B,T,C
        return out


class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    """ Linear layer followed by nonlinearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class SmallLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size,n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets==None:
            loss=None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop to get block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


# Minibatch Function
def get_batch(split):
    data = train_data if split == 'train' else test_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(m):
    out = {}
    m.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = m(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out

In [ ]:
# Hyperparams
batch_size = 64 
block_size = 128
max_iters = 1
eval_interval = 1000
eval_iters = 100
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
n_embd = 256
n_head = 8
n_layer = 6
dropout = 0.2

# Construct Model
model = SmallLanguageModel()
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
# Train Model
for steps in range(max_iters):

    # Occasionally evaluate the loss on train and val sets
    if steps % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {steps}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    # Sample
    xb, yb = get_batch('train')

    losses.backward()

    # Evaluate Loss
    logits, loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
sys.path.append('/content/drive/MyDrive/shapely/code')

MODEL_PATH = '/content/drive/MyDrive/shapely/code/SLM.py'
from SLM import SmallLanguageModel
m = SmallLanguageModel()

m.generate()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/shapely/data/shakespeare.txt'